# 09 — The Softmax Activation Function

In the previous notebooks we:

- Built dense (fully–connected) layers  
- Used the ReLU activation to introduce non-linearity  
- Stacked layers to form a small network

For **classification**, though, ReLU on the output layer isn’t ideal:

- ReLU outputs are **unbounded** (can grow arbitrarily large)
- They’re **not normalized** (no obvious notion of “probability”)
- Each output is **independent** of the others

For classifiers we usually want:

- A vector of **non-negative** numbers  
- That **sum to 1**  
- Interpretable as **class probabilities**

Softmax does exactly this: it takes raw scores (often called *logits*) and turns them into a probability distribution.


In [ ]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()

## 1. From raw outputs to probabilities (plain Python)

Imagine a layer that produced these 3 outputs for a single sample:

```python
layer_outputs = [4.8, 1.21, 2.385]
```

By themselves, those numbers have no probabilistic meaning. Softmax first **exponentiates** them using Euler's number $e \approx 2.71828$:

$$e^{z_j}$$

for each output $z_j$.

Let’s do this step-by-step with basic Python.


In [ ]:
# Raw layer outputs (logits)
layer_outputs = [4.8, 1.21, 2.385]

# Euler's number (could also use math.e)
E = 2.71828182846

exp_values = []
for output in layer_outputs:
    exp_values.append(E ** output)  # exponentiation

print("Exponentiated values:")
print(exp_values)

These exponentiated values are:

- Always **non-negative**
- Preserve ordering (if $z_1 > z_2$ then $e^{z_1} > e^{z_2}$)

Next, Softmax **normalizes** them so they sum to 1:

$$p_j = \frac{e^{z_j}}{\sum_k e^{z_k}}$$

This gives us a proper probability distribution.


In [ ]:
# Sum of exponentiated values (denominator)
norm_base = sum(exp_values)

norm_values = []
for value in exp_values:
    norm_values.append(value / norm_base)

print("Normalized exponentiated values (probabilities):")
print(norm_values)
print("Sum of probabilities:", sum(norm_values))

## 2. NumPy implementation for a single sample

Doing this manually works, but is clunky. NumPy can apply the same operations in a vectorized way.


In [ ]:
layer_outputs = np.array([4.8, 1.21, 2.385])

# Exponentiate
exp_values = np.exp(layer_outputs)
print("Exponentiated values:")
print(exp_values)

# Normalize
norm_values = exp_values / np.sum(exp_values)
print("\nNormalized exponentiated values (probabilities):")
print(norm_values)
print("Sum of probabilities:", np.sum(norm_values))

## 3. Batches and the `axis` argument

During training we usually process **batches** of samples. Suppose a layer output is:

$$
\begin{bmatrix}
4.8 & 1.21 & 2.385 \\
8.9 & -1.81 & 0.2 \\
1.41 & 1.051 & 0.026
\end{bmatrix}
$$

Each **row** is one sample, each **column** is one neuron.

To normalize *per sample*, we need the sum across **columns** (axis 1) for each row.


In [ ]:
layer_outputs = np.array([
    [4.8, 1.21, 2.385],
    [8.9, -1.81, 0.2],
    [1.41, 1.051, 0.026],
])

print("Layer outputs:\n", layer_outputs)

print("\nSum without axis (all elements):")
print(np.sum(layer_outputs))

print("\nSum along axis=0 (column-wise):")
print(np.sum(layer_outputs, axis=0))

print("\nSum along axis=1 (row-wise):")
print(np.sum(layer_outputs, axis=1))

print("\nSum along axis=1 with keepdims=True (column vector):")
row_sums = np.sum(layer_outputs, axis=1, keepdims=True)
print(row_sums)
print("Shape:", row_sums.shape)

`keepdims=True` keeps the result 2-D (shape `(n_samples, 1)`), which is perfect for broadcasting:

- `layer_outputs` has shape `(n_samples, n_neurons)`  
- `row_sums` has shape `(n_samples, 1)`

Dividing them gives **row-wise normalization**.


In [ ]:
# Exponentiate per element
exp_values = np.exp(layer_outputs)

# Row sums as column vector
row_sums = np.sum(exp_values, axis=1, keepdims=True)

# Normalize to probabilities
probabilities = exp_values / row_sums

print("Probabilities:\n", probabilities)
print("Row sums:", np.sum(probabilities, axis=1))

## 4. A Softmax activation class (with numerical stability)

The full Softmax forward pass for a batch:

$$
\text{softmax}(z)_j = \frac{e^{z_j}}{\sum_k e^{z_k}}
$$

One big practical trick: **subtract the maximum value per row before exponentiating**:

$$
\tilde{z}_j = z_j - \max_k z_k
$$

This keeps the largest value at 0 and all others $\le 0$, which helps prevent `exp` from overflowing.

Let’s implement this as a reusable activation class.


In [ ]:
class Activation_Softmax:
    # Forward pass
    def forward(self, inputs):
        # Save input for potential backward pass later
        self.inputs = inputs

        # Subtract max per row for numerical stability
        shifted_inputs = inputs - np.max(inputs, axis=1, keepdims=True)

        # Exponentiate
        exp_values = np.exp(shifted_inputs)

        # Normalize per sample
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)

        self.output = probabilities

### 4.1. Subtracting a constant doesn’t change the probabilities

Softmax is invariant to adding (or subtracting) the same constant to all logits for a sample.

Let’s verify it numerically.


In [ ]:
softmax = Activation_Softmax()

# Original logits
softmax.forward(np.array([[1.0, 2.0, 3.0]]))
print("Softmax([1, 2, 3]):")
print(softmax.output)

# Shifted logits (subtract 3 from each)
softmax.forward(np.array([[-2.0, -1.0, 0.0]]))
print("\nSoftmax([-2, -1, 0]) (shifted by -3):")
print(softmax.output)

### 4.2. Scaling *does* change the distribution

Dividing logits by 2 changes how “peaked” the distribution is, because the exponential is nonlinear.


In [ ]:
softmax.forward(np.array([[0.5, 1.0, 1.5]]))
print("Softmax([0.5, 1.0, 1.5]):")
print(softmax.output)

## 5. Putting it all together with Dense + ReLU + Softmax

Now let’s re-use dense and ReLU layers and add Softmax as the **output layer activation** for a tiny 3-class classifier on the spiral dataset.


In [ ]:
# Dense (fully-connected) layer
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        # Small random weights and zero biases
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases


# ReLU activation
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

In [ ]:
# Create dataset
X, y = spiral_data(samples=100, classes=3)

# First dense layer: 2 inputs -> 3 neurons
dense1 = Layer_Dense(2, 3)
activation1 = Activation_ReLU()

# Second dense layer: 3 inputs -> 3 neurons (one per class)
dense2 = Layer_Dense(3, 3)
activation2 = Activation_Softmax()

# Forward pass
dense1.forward(X)
activation1.forward(dense1.output)

dense2.forward(activation1.output)
activation2.forward(dense2.output)

# Look at the first few outputs
print("First 5 softmax outputs (probability distributions):")
print(activation2.output[:5])

We should see:

- Each row has 3 values  
- Each row sums to 1  
- At this stage (random weights), the distribution is close to uniform (~0.33 per class), because the network hasn’t been trained yet

Let’s check the row sums and predicted classes.


In [ ]:
# Check that each row sums to ~1
row_sums = np.sum(activation2.output, axis=1)
print("Row sums (first 5):", row_sums[:5])

# Predicted classes = index of highest probability
predictions = np.argmax(activation2.output, axis=1)

print("Predicted classes (first 10):", predictions[:10])
print("True classes      (first 10):", y[:10])

## 6. Summary

- Softmax converts raw logits into a **probability distribution**:
  - Non-negative  
  - Sums to 1 per sample  
- It is built from:
  1. Exponentiation: $e^{z_j}$  
  2. Normalization: divide by $\sum_k e^{z_k}$  
- We use `axis=1` and `keepdims=True` to normalize **per sample** in a batch.  
- Subtracting the maximum logit per row before exponentiating prevents overflow but **does not** change the final probabilities.  
- Using Softmax on the output layer turns our network into a proper **classifier**, giving both predicted classes and meaningful **confidence scores**.

In later notebooks we’ll add a **loss function** (categorical cross-entropy) and actually **train** the network so those probabilities become informative instead of random.
